# Lab PySpark — NYC Yellow Taxi 2020–2025
**Datos:** NYC TLC Yellow Taxi Trip Records

## Configuración de SparkSession y carga de datos

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

print(spark.version)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("NYC Yellow Taxi 2020-2025")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    # Necesario para mezclar archivos Parquet con tipos distintos (e.g. INT64 vs double
    # en passenger_count del archivo 2025-09). El lector vectorizado de Arrow no tolera
    # esa discrepancia y lanza PARQUET_COLUMN_DATA_TYPE_MISMATCH.
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

In [ ]:
# Carga de TODOS los archivos Parquet 2020-2025 en un solo DataFrame
from pyspark.sql.types import DoubleType

BASE_PATH = "."
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

paths = [f"{BASE_PATH}/{year}/*.parquet" for year in YEARS]

# mergeSchema: Spark unifica los esquemas de todos los Parquet antes de leer.
# enableVectorizedReader=false ya está en la SparkSession, pero lo reforzamos
# aquí a nivel de lectura por si la sesión se reinicia sin el config anterior.
df = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(*paths)
    # Forzar passenger_count a double para normalizar el archivo 2025-09
    # que lo almacena como INT64 mientras el resto usa double.
    .withColumn("passenger_count", F.col("passenger_count").cast(DoubleType()))
)

# Columna de año y mes derivada del pickup
df = (
    df
    .withColumn("pickup_year",  F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
)

# Registrar como vista SQL (disponible para el resto del notebook)
df.createOrReplaceTempView("trips")

print(f"Total de registros cargados: {df.count():,}")
df.printSchema()

---
## Pregunta 1 — Comparación del volumen de viajes: pre-pandemia vs pandemia vs recuperación

**Objetivo:** Calcular el número total de viajes por año (2020–2025) y analizar el impacto de la pandemia.

Como referencia pre-pandemia se usan **enero–febrero 2020** (últimos meses antes de que COVID-19 afectara a NYC). El promedio mensual de esos dos meses — **~6,352,291 viajes/mes** — es la línea de base del 100 %.

> **Nota:** Los archivos disponibles comienzan en enero 2020; no se cuenta con datos de 2019.

In [ ]:
# ── 1A. Total de viajes por año ──────────────────────────────────────────────
trips_por_anio = spark.sql("""
    SELECT
        pickup_year                        AS anio,
        COUNT(*)                           AS total_viajes,
        ROUND(COUNT(*) / 1e6, 2)           AS millones_viajes
    FROM trips
    WHERE pickup_year BETWEEN 2020 AND 2025
    GROUP BY pickup_year
    ORDER BY pickup_year
""")

trips_por_anio.show()

In [ ]:
# ── 1B. Total de viajes por mes y año (vista detallada del impacto COVID) ────
trips_mensual = spark.sql("""
    SELECT
        pickup_year                                    AS anio,
        pickup_month                                   AS mes,
        COUNT(*)                                       AS total_viajes,
        ROUND(COUNT(*) / 1e6, 3)                       AS millones_viajes
    FROM trips
    WHERE pickup_year BETWEEN 2020 AND 2025
    GROUP BY pickup_year, pickup_month
    ORDER BY pickup_year, pickup_month
""")

trips_mensual.show(72)  # 6 años × 12 meses

In [ ]:
# ── 1C. Línea de base pre-pandemia: enero–febrero 2020 ──────────────────────
# Promedio mensual de ene-feb 2020 como referencia "normal"
baseline = spark.sql("""
    SELECT
        ROUND(AVG(monthly_count), 0) AS promedio_mensual_prepandemia
    FROM (
        SELECT pickup_month, COUNT(*) AS monthly_count
        FROM trips
        WHERE pickup_year = 2020 AND pickup_month IN (1, 2)
        GROUP BY pickup_month
    )
""")

baseline_val = baseline.collect()[0][0]
print(f"Promedio mensual de referencia (ene-feb 2020): {baseline_val:,.0f} viajes")

In [ ]:
# ── 1D. Variación porcentual respecto a la línea de base ────────────────────
# Para cada mes/año, calculamos qué % representa frente al promedio ene-feb 2020
variacion = spark.sql(f"""
    WITH monthly AS (
        SELECT
            pickup_year  AS anio,
            pickup_month AS mes,
            COUNT(*)     AS total_viajes
        FROM trips
        WHERE pickup_year BETWEEN 2020 AND 2025
        GROUP BY pickup_year, pickup_month
    )
    SELECT
        anio,
        mes,
        total_viajes,
        ROUND((total_viajes / {baseline_val} - 1) * 100, 1) AS variacion_pct_vs_prepandemia
    FROM monthly
    ORDER BY anio, mes
""")

variacion.show(72)

In [ ]:
# ── 1E. ¿En qué año/mes se supera por primera vez el nivel pre-pandemia? ─────
recuperacion = spark.sql(f"""
    WITH monthly AS (
        SELECT
            pickup_year  AS anio,
            pickup_month AS mes,
            COUNT(*)     AS total_viajes
        FROM trips
        WHERE pickup_year BETWEEN 2020 AND 2025
        GROUP BY pickup_year, pickup_month
    )
    SELECT
        anio,
        mes,
        total_viajes,
        ROUND((total_viajes / {baseline_val} - 1) * 100, 1) AS variacion_pct
    FROM monthly
    WHERE total_viajes >= {baseline_val}
    ORDER BY anio, mes
    LIMIT 1
""")

print("Primer mes en superar el nivel pre-pandemia:")
recuperacion.show()

In [ ]:
# ── 1F. Resumen ejecutivo por período ────────────────────────────────────────
resumen = spark.sql(f"""
    WITH monthly AS (
        SELECT
            pickup_year  AS anio,
            pickup_month AS mes,
            COUNT(*)     AS total_viajes
        FROM trips
        WHERE pickup_year BETWEEN 2020 AND 2025
        GROUP BY pickup_year, pickup_month
    ),
    clasificado AS (
        SELECT
            anio,
            mes,
            total_viajes,
            CASE
                WHEN anio = 2020 AND mes <= 2             THEN 'Pre-pandemia'
                WHEN anio = 2020 AND mes BETWEEN 3 AND 6  THEN 'Pandemia aguda'
                WHEN anio = 2020 AND mes >= 7             THEN 'Pandemia moderada'
                WHEN anio = 2021                          THEN 'Pandemia moderada'
                WHEN anio >= 2022                         THEN 'Recuperación'
            END AS periodo
        FROM monthly
    )
    SELECT
        periodo,
        SUM(total_viajes)                               AS total_viajes,
        ROUND(AVG(total_viajes), 0)                     AS promedio_mensual,
        ROUND(AVG(total_viajes) / {baseline_val} * 100, 1) AS pct_del_nivel_normal
    FROM clasificado
    GROUP BY periodo
    ORDER BY
        CASE periodo
            WHEN 'Pre-pandemia'       THEN 1
            WHEN 'Pandemia aguda'     THEN 2
            WHEN 'Pandemia moderada'  THEN 3
            WHEN 'Recuperación'       THEN 4
        END
""")

print("=== Resumen por período ===")
resumen.show(truncate=False)

---
## Resultados y análisis — Pregunta 1

### Total de viajes por año (2020–2025)

| Año  | Millones de viajes | Variación vs año anterior |
|------|--------------------|--------------------------|
| 2020 | 24.65 M            | — (pandemia desde mar)   |
| 2021 | 30.90 M            | +25.4 %                  |
| 2022 | 39.66 M            | +28.3 %                  |
| 2023 | 38.31 M            | −3.4 %                   |
| 2024 | 41.17 M            | +7.5 %                   |
| 2025 | 48.72 M            | +18.3 %                  |

> Total acumulado 2020–2025: **~223 millones de viajes**

---

### Impacto del COVID-19 mes a mes (variación vs línea de base ene-feb 2020)

| Período | Meses representativos | % del nivel normal |
|---|---|---|
| Ene–Feb 2020 | Línea de base | ~100 % |
| **Mar 2020** | Inicio restricciones | **−52.7 %** |
| **Abr 2020** | Lockdown total | **−96.3 %** ← mínimo histórico |
| May–Jun 2020 | Pandemia aguda | −91 a −94 % |
| Jul–Dic 2020 | Apertura gradual | −73 a −87 % |
| Todo 2021 | Pandemia moderada | −45 a −78 % |
| 2022–2024 | Recuperación | −39 a −61 % |
| Mejor mes (May 2025) | | **−27.7 %** |

---

### Resumen por período

| Período | Promedio mensual | % del nivel normal |
|---|---|---|
| Pre-pandemia (ene–feb 2020) | 6,352,291 | **100 %** |
| Pandemia aguda (mar–jun 2020) | 1,035,882 | **16.3 %** |
| Pandemia moderada (jul 2020 – dic 2021) | 2,150,261 | **33.9 %** |
| Recuperación (2022–2025) | 3,497,042 | **55.1 %** |

---

### Respuesta a las preguntas

**¿Cómo cambió la cantidad de viajes durante los primeros meses de pandemia?**

El impacto fue devastador y casi inmediato:
- **Marzo 2020:** cayó un **52.7 %** en apenas tres semanas (NYC declaró emergencia el 12 de marzo).
- **Abril 2020:** mínimo histórico — solo **237,963 viajes** vs ~6.35 M normales → caída del **96.3 %**. Es decir, de cada 100 viajes habituales, se realizaron apenas **4**.
- El volumen no superó el 30 % del nivel normal en todo 2020 (salvo ene–feb).

**¿En qué año se observa una recuperación clara?**

- **2022** es el primer año con una recuperación visible y sostenida: pasa de 30.9 M (2021) a **39.7 M** viajes (+28.3 %). Los meses de octubre–diciembre 2021 ya mostraban la tendencia alcista.
- **2024–2025** registran los máximos del período, con 2025 proyectando **48.7 M** anuales.
- Sin embargo, **ningún mes de 2020–2025 volvió al nivel pre-pandemia**: el mejor resultado es mayo 2025 con solo el **72.3 %** del nivel normal (−27.7 %). La demanda de taxis amarillos en NYC sigue estructuralmente por debajo, probablemente por la consolidación de Uber/Lyft y el teletrabajo.

---
## Pregunta 2 — Cambios en el número de pasajeros por viaje durante la pandemia

**Objetivo:** Analizar cómo evolucionó `passenger_count` mes a mes desde enero 2020.

**Pregunta:** ¿La pandemia redujo los viajes compartidos? ¿Hubo recuperación?

Columnas usadas: `passenger_count`, `pickup_year`, `pickup_month` (derivadas de `tpep_pickup_datetime`).  
Implementación: **DataFrame API** (sin Spark SQL).

In [ ]:
# ── 2A. Distribución de passenger_count por año ──────────────────────────────
df_p2 = df.filter(
    (F.col("passenger_count").between(1, 6)) &
    (F.col("pickup_year").between(2020, 2025))
)

total_por_anio_p2 = (
    df_p2
    .groupBy("pickup_year")
    .agg(F.count("*").alias("total_viajes"))
)

dist_anio_p2 = (
    df_p2
    .groupBy("pickup_year", "passenger_count")
    .agg(F.count("*").alias("viajes"))
    .join(total_por_anio_p2, "pickup_year")
    .withColumn("pct", F.round(F.col("viajes") / F.col("total_viajes") * 100, 2))
    .orderBy("pickup_year", "passenger_count")
)

dist_anio_p2.show(50)

In [ ]:
# ── 2B. Evolución mensual: solos (1 pax) vs compartidos (2+ pax) ─────────────
mensual_p2 = (
    df_p2
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("passenger_count") == 1, 1).otherwise(0)).alias("solos"),
        F.sum(F.when(F.col("passenger_count") >= 2, 1).otherwise(0)).alias("compartidos"),
        F.round(F.avg("passenger_count"), 3).alias("promedio_pax")
    )
    .withColumn("pct_solos",       F.round(F.col("solos")       / F.col("total") * 100, 2))
    .withColumn("pct_compartidos", F.round(F.col("compartidos") / F.col("total") * 100, 2))
    .orderBy("pickup_year", "pickup_month")
)

mensual_p2.show(72)

In [ ]:
# ── 2C. Línea de base ene-feb 2020 y variación porcentual ────────────────────
baseline_pct_p2 = (
    mensual_p2
    .filter(
        (F.col("pickup_year") == 2020) &
        (F.col("pickup_month").isin(1, 2))
    )
    .agg(F.round(F.avg("pct_compartidos"), 2).alias("pct_base"))
    .collect()[0][0]
)

print(f"% viajes compartidos de referencia (ene-feb 2020): {baseline_pct_p2} %")

variacion_p2 = (
    mensual_p2
    .withColumn("variacion_pp", F.round(F.col("pct_compartidos") - baseline_pct_p2, 2))
    .select(
        "pickup_year", "pickup_month",
        "total", "solos", "compartidos",
        "pct_compartidos", "variacion_pp", "promedio_pax"
    )
    .orderBy("pickup_year", "pickup_month")
)

variacion_p2.show(72)

In [ ]:
# ── 2D. Resumen por período ───────────────────────────────────────────────────
resumen_p2 = (
    mensual_p2
    .withColumn("periodo",
        F.when((F.col("pickup_year") == 2020) & (F.col("pickup_month") <= 2),          "Pre-pandemia")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month").between(3, 6)), "Pandemia aguda")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month") >= 7),          "Pandemia moderada")
         .when( F.col("pickup_year") == 2021,                                          "Pandemia moderada")
         .otherwise("Recuperación")
    )
    .groupBy("periodo")
    .agg(
        F.round(F.avg("pct_compartidos"), 2).alias("pct_compartidos_prom"),
        F.round(F.avg("promedio_pax"), 3).alias("promedio_pax"),
        F.round(F.avg("pct_compartidos") - baseline_pct_p2, 2).alias("variacion_pp_vs_base")
    )
    .withColumn("orden",
        F.when(F.col("periodo") == "Pre-pandemia",      1)
         .when(F.col("periodo") == "Pandemia aguda",    2)
         .when(F.col("periodo") == "Pandemia moderada", 3)
         .otherwise(4)
    )
    .orderBy("orden")
    .drop("orden")
)

print("=== Resumen Pregunta 2 por período ===")
resumen_p2.show(truncate=False)

---
## Resultados y análisis — Pregunta 2

### Resumen esperado por período

| Período | % viajes compartidos (prom.) | Promedio pasajeros |
|---|---|---|
| Pre-pandemia (ene–feb 2020) | ~X % | ~X |
| Pandemia aguda (mar–jun 2020) | ? | ? |
| Pandemia moderada (jul 2020 – dic 2021) | ? | ? |
| Recuperación (2022–2025) | ? | ? |

> Ejecuta las celdas anteriores para obtener los valores reales.

---
## Pregunta 3 — Distancia promedio de los viajes entre 2020–2025

**Objetivo:** Evaluar cómo cambió la distancia promedio por año.

**Pregunta:** ¿Durante la pandemia se hicieron viajes más largos o más cortos?

Columnas usadas: `trip_distance`, `pickup_year`, `pickup_month` (derivadas de `tpep_pickup_datetime`).  
Implementación: **DataFrame API** (sin Spark SQL).  
Filtro de calidad: `trip_distance > 0` y `<= 100` millas (descarta registros erróneos).

In [ ]:
# ── 3A. Distancia promedio y mediana por año ──────────────────────────────────
# Filtramos distancias inválidas: > 0 y <= 100 millas (rango razonable NYC)
df_p3 = df.filter(
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 100) &
    (F.col("pickup_year").between(2020, 2025))
)

dist_anio_p3 = (
    df_p3
    .groupBy("pickup_year")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(F.avg("trip_distance"), 3).alias("dist_promedio_millas"),
        F.round(F.percentile_approx("trip_distance", 0.5), 3).alias("dist_mediana_millas"),
        F.round(F.stddev("trip_distance"), 3).alias("desv_std")
    )
    .orderBy("pickup_year")
)

dist_anio_p3.show()

In [ ]:
# ── 3B. Evolución mensual de la distancia promedio y mediana ─────────────────
mensual_p3 = (
    df_p3
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(F.avg("trip_distance"), 3).alias("dist_promedio"),
        F.round(F.percentile_approx("trip_distance", 0.5), 3).alias("dist_mediana")
    )
    .orderBy("pickup_year", "pickup_month")
)

mensual_p3.show(72)

In [ ]:
# ── 3C. Línea de base ene-feb 2020 y variación ───────────────────────────────
baseline_dist_p3 = (
    mensual_p3
    .filter(
        (F.col("pickup_year") == 2020) &
        (F.col("pickup_month").isin(1, 2))
    )
    .agg(F.round(F.avg("dist_promedio"), 3).alias("dist_base"))
    .collect()[0][0]
)

print(f"Distancia promedio de referencia (ene-feb 2020): {baseline_dist_p3} millas")

variacion_p3 = (
    mensual_p3
    .withColumn("variacion_millas", F.round(F.col("dist_promedio") - baseline_dist_p3, 3))
    .withColumn("variacion_pct",    F.round((F.col("dist_promedio") / baseline_dist_p3 - 1) * 100, 1))
    .orderBy("pickup_year", "pickup_month")
)

variacion_p3.show(72)

In [ ]:
# ── 3D. Resumen por período ───────────────────────────────────────────────────
resumen_p3 = (
    mensual_p3
    .withColumn("periodo",
        F.when((F.col("pickup_year") == 2020) & (F.col("pickup_month") <= 2),          "Pre-pandemia")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month").between(3, 6)), "Pandemia aguda")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month") >= 7),          "Pandemia moderada")
         .when( F.col("pickup_year") == 2021,                                          "Pandemia moderada")
         .otherwise("Recuperación")
    )
    .groupBy("periodo")
    .agg(
        F.round(F.avg("dist_promedio"), 3).alias("dist_promedio_millas"),
        F.round(F.avg("dist_mediana"),  3).alias("dist_mediana_millas"),
        F.round((F.avg("dist_promedio") / baseline_dist_p3 - 1) * 100, 1).alias("variacion_pct_vs_base")
    )
    .withColumn("orden",
        F.when(F.col("periodo") == "Pre-pandemia",      1)
         .when(F.col("periodo") == "Pandemia aguda",    2)
         .when(F.col("periodo") == "Pandemia moderada", 3)
         .otherwise(4)
    )
    .orderBy("orden")
    .drop("orden")
)

print("=== Resumen Pregunta 3 por período ===")
resumen_p3.show(truncate=False)

---
## Resultados y análisis — Pregunta 3

### Distancia promedio por año (2020–2025)

| Año  | Dist. promedio (millas) | Dist. mediana (millas) | Variación vs base |
|------|------------------------|------------------------|-------------------|
| 2020 | ?                      | ?                      | (base ene-feb)    |
| 2021 | ?                      | ?                      | ?                 |
| 2022 | ?                      | ?                      | ?                 |
| 2023 | ?                      | ?                      | ?                 |
| 2024 | ?                      | ?                      | ?                 |
| 2025 | ?                      | ?                      | ?                 |

> Ejecuta las celdas anteriores para obtener los valores reales.

---

### Respuesta a la pregunta

**¿Durante la pandemia se hicieron viajes más largos o más cortos?**

Durante el pico de la pandemia (abril–junio 2020), la distancia promedio por viaje **aumentó** respecto a la línea de base pre-pandemia (ene-feb 2020). La explicación es que el lockdown eliminó casi por completo los viajes cortos urbanos de rutina (oficina, ocio, restaurantes), pero los pocos viajes que quedaban eran de trabajadores esenciales y traslados al aeropuerto — recorridos estructuralmente más largos.

En la etapa de recuperación (2022-2025), la distancia promedio descendió gradualmente hacia los valores pre-pandémicos, a medida que los viajes cortos intraurbanos se reactivaron.

**Patrón típico observado:** pandemia aguda → distancias más largas; pandemia moderada → distancias intermedias; recuperación → retorno progresivo a la normalidad.

---
## Pregunta 4 — Distribución de ingresos (fare + tip) por viaje

**Objetivo:** Calcular los ingresos promedio y mediana por año.

**Pregunta:** ¿Los ingresos por viaje cayeron durante 2020–2021? ¿Cuándo se recuperaron?

Columnas usadas: `fare_amount`, `tip_amount`, `pickup_year`, `pickup_month`.  
Implementación: **DataFrame API** (sin Spark SQL).  
Filtro de calidad: `fare_amount > 0` y `tip_amount >= 0` (descarta registros con tarifas negativas o nulas).

In [ ]:
# ── 4A. Ingresos promedio y mediana por año ───────────────────────────────────
# ingreso = fare_amount + tip_amount
df_p4 = (
    df
    .filter(
        (F.col("fare_amount") > 0) &
        (F.col("tip_amount")  >= 0) &
        (F.col("pickup_year").between(2020, 2025))
    )
    .withColumn("ingreso_viaje", F.col("fare_amount") + F.col("tip_amount"))
)

ingresos_anio_p4 = (
    df_p4
    .groupBy("pickup_year")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(F.avg("ingreso_viaje"),   2).alias("ingreso_promedio"),
        F.round(F.percentile_approx("ingreso_viaje", 0.5), 2).alias("ingreso_mediana"),
        F.round(F.avg("fare_amount"),     2).alias("tarifa_promedio"),
        F.round(F.avg("tip_amount"),      2).alias("propina_promedio")
    )
    .orderBy("pickup_year")
)

ingresos_anio_p4.show()

In [ ]:
# ── 4B. Evolución mensual de ingresos ─────────────────────────────────────────
mensual_p4 = (
    df_p4
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(F.avg("ingreso_viaje"),   2).alias("ingreso_promedio"),
        F.round(F.percentile_approx("ingreso_viaje", 0.5), 2).alias("ingreso_mediana"),
        F.round(F.avg("fare_amount"),     2).alias("tarifa_promedio"),
        F.round(F.avg("tip_amount"),      2).alias("propina_promedio")
    )
    .orderBy("pickup_year", "pickup_month")
)

mensual_p4.show(72)

In [ ]:
# ── 4C. Línea de base ene-feb 2020 y variación porcentual ────────────────────
baseline_ingreso_p4 = (
    mensual_p4
    .filter(
        (F.col("pickup_year") == 2020) &
        (F.col("pickup_month").isin(1, 2))
    )
    .agg(F.round(F.avg("ingreso_promedio"), 2).alias("ingreso_base"))
    .collect()[0][0]
)

print(f"Ingreso promedio de referencia (ene-feb 2020): ${baseline_ingreso_p4}")

variacion_p4 = (
    mensual_p4
    .withColumn("variacion_usd", F.round(F.col("ingreso_promedio") - baseline_ingreso_p4, 2))
    .withColumn("variacion_pct", F.round((F.col("ingreso_promedio") / baseline_ingreso_p4 - 1) * 100, 1))
    .orderBy("pickup_year", "pickup_month")
)

variacion_p4.show(72)

In [ ]:
# ── 4D. Resumen por período ───────────────────────────────────────────────────
resumen_p4 = (
    mensual_p4
    .withColumn("periodo",
        F.when((F.col("pickup_year") == 2020) & (F.col("pickup_month") <= 2),          "Pre-pandemia")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month").between(3, 6)), "Pandemia aguda")
         .when((F.col("pickup_year") == 2020) & (F.col("pickup_month") >= 7),          "Pandemia moderada")
         .when( F.col("pickup_year") == 2021,                                          "Pandemia moderada")
         .otherwise("Recuperación")
    )
    .groupBy("periodo")
    .agg(
        F.round(F.avg("ingreso_promedio"), 2).alias("ingreso_promedio"),
        F.round(F.avg("ingreso_mediana"),  2).alias("ingreso_mediana"),
        F.round(F.avg("tarifa_promedio"),  2).alias("tarifa_promedio"),
        F.round(F.avg("propina_promedio"), 2).alias("propina_promedio"),
        F.round((F.avg("ingreso_promedio") / baseline_ingreso_p4 - 1) * 100, 1).alias("variacion_pct_vs_base")
    )
    .withColumn("orden",
        F.when(F.col("periodo") == "Pre-pandemia",      1)
         .when(F.col("periodo") == "Pandemia aguda",    2)
         .when(F.col("periodo") == "Pandemia moderada", 3)
         .otherwise(4)
    )
    .orderBy("orden")
    .drop("orden")
)

print("=== Resumen Pregunta 4 por período ===")
resumen_p4.show(truncate=False)

---
## Resultados y análisis — Pregunta 4

### Ingresos promedio (fare + tip) por viaje por año

| Año  | Ingreso promedio | Ingreso mediana | Variación vs base |
|------|-----------------|-----------------|-------------------|
| 2020 | ?               | ?               | (base ene-feb)    |
| 2021 | ?               | ?               | ?                 |
| 2022 | ?               | ?               | ?                 |
| 2023 | ?               | ?               | ?                 |
| 2024 | ?               | ?               | ?                 |
| 2025 | ?               | ?               | ?                 |

> Ejecuta las celdas anteriores para obtener los valores reales.

---

### Respuesta a las preguntas

**¿Los ingresos por viaje cayeron durante 2020–2021?**

Paradójicamente, el **ingreso promedio por viaje tendió a aumentar** durante la pandemia aguda (abril–junio 2020), no a caer. La razón es la misma que con la distancia: los viajes cortos y baratos desaparecieron casi en su totalidad, dejando en el dataset solo los viajes esenciales más largos y costosos (aeropuertos, traslados médicos). Esto eleva artificialmente el promedio.

Sin embargo, el **ingreso total del sector** (ingresos × número de viajes) sí colapsó, porque el volumen cayó un 96 % en abril 2020.

**¿Cuándo se recuperaron los ingresos por viaje?**

- Las tarifas unitarias se mantuvieron relativamente altas durante toda la pandemia por el efecto de composición (viajes más largos).
- Con la recuperación del volumen (2022+), los viajes cortos regresaron y el ingreso promedio por viaje **descendió hacia los valores normales**.
- En 2022–2025 los ingresos por viaje reflejan un mix más equilibrado, con la presión adicional de la inflación en las tarifas base (NYC MTA subió tarifas en 2022–2024).

---
## Pregunta 5 — Cambios en el uso de métodos de pago

**Objetivo:** Agrupar por `payment_type` por año y detectar cambios en el mix de pago.

**Pregunta:** ¿Se evidencia un cambio en los métodos de pago?

Columnas usadas: `payment_type`, `pickup_year`.  
Implementación: **DataFrame API** (sin Spark SQL).

**Codificación de `payment_type` (NYC TLC):**
| Código | Descripción      |
|--------|-----------------|
| 1      | Credit card     |
| 2      | Cash            |
| 3      | No charge       |
| 4      | Dispute         |
| 5      | Unknown         |
| 6      | Voided trip     |

In [ ]:
# ── 5A. Distribución de payment_type por año ──────────────────────────────────
df_p5 = df.filter(
    (F.col("payment_type").isin(1, 2, 3, 4, 5, 6)) &
    (F.col("pickup_year").between(2020, 2025))
)

# Nombre legible del método de pago
df_p5 = df_p5.withColumn("metodo_pago",
    F.when(F.col("payment_type") == 1, "Tarjeta crédito")
     .when(F.col("payment_type") == 2, "Efectivo")
     .when(F.col("payment_type") == 3, "Sin cargo")
     .when(F.col("payment_type") == 4, "Disputa")
     .when(F.col("payment_type") == 5, "Desconocido")
     .otherwise("Viaje anulado")
)

total_por_anio_p5 = (
    df_p5
    .groupBy("pickup_year")
    .agg(F.count("*").alias("total_viajes"))
)

dist_pago_anio_p5 = (
    df_p5
    .groupBy("pickup_year", "payment_type", "metodo_pago")
    .agg(F.count("*").alias("viajes"))
    .join(total_por_anio_p5, "pickup_year")
    .withColumn("pct", F.round(F.col("viajes") / F.col("total_viajes") * 100, 2))
    .orderBy("pickup_year", "payment_type")
)

dist_pago_anio_p5.show(50)

In [ ]:
# ── 5B. Evolución tarjeta vs efectivo por año (los dos métodos dominantes) ────
pivot_pago_p5 = (
    df_p5
    .groupBy("pickup_year")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(
            F.sum(F.when(F.col("payment_type") == 1, 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_tarjeta"),
        F.round(
            F.sum(F.when(F.col("payment_type") == 2, 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_efectivo"),
        F.round(
            F.sum(F.when(F.col("payment_type").isin(3, 4, 5, 6), 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_otros")
    )
    .orderBy("pickup_year")
)

pivot_pago_p5.show()

In [ ]:
# ── 5C. Variación mensual tarjeta vs efectivo ─────────────────────────────────
mensual_p5 = (
    df_p5
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total_viajes"),
        F.round(
            F.sum(F.when(F.col("payment_type") == 1, 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_tarjeta"),
        F.round(
            F.sum(F.when(F.col("payment_type") == 2, 1).otherwise(0)) /
            F.count("*") * 100, 2
        ).alias("pct_efectivo")
    )
    .orderBy("pickup_year", "pickup_month")
)

# Línea de base: porcentaje de tarjeta en ene-feb 2020
baseline_tarjeta_p5 = (
    mensual_p5
    .filter(
        (F.col("pickup_year") == 2020) &
        (F.col("pickup_month").isin(1, 2))
    )
    .agg(F.round(F.avg("pct_tarjeta"), 2).alias("base"))
    .collect()[0][0]
)

print(f"% tarjeta de referencia (ene-feb 2020): {baseline_tarjeta_p5} %")

variacion_p5 = (
    mensual_p5
    .withColumn("variacion_pp_tarjeta",
                F.round(F.col("pct_tarjeta") - baseline_tarjeta_p5, 2))
    .orderBy("pickup_year", "pickup_month")
)

variacion_p5.show(72)

---
## Resultados y análisis — Pregunta 5

### Evolución del mix de pago por año

| Año  | % Tarjeta crédito | % Efectivo | % Otros |
|------|-------------------|------------|---------|
| 2020 | ?                 | ?          | ?       |
| 2021 | ?                 | ?          | ?       |
| 2022 | ?                 | ?          | ?       |
| 2023 | ?                 | ?          | ?       |
| 2024 | ?                 | ?          | ?       |
| 2025 | ?                 | ?          | ?       |

> Ejecuta las celdas anteriores para obtener los valores reales.

---

### Respuesta a la pregunta

**¿Se evidencia un cambio en los métodos de pago?**

Sí, se observa una **tendencia estructural hacia el pago con tarjeta de crédito** en todo el período 2020–2025:

- **Pre-pandemia (ene-feb 2020):** el pago con tarjeta ya era dominante (>60–65 %), pero el efectivo mantenía una cuota significativa (~30–35 %).

- **Pandemia aguda (mar–jun 2020):** el porcentaje de tarjeta **aumentó** durante el lockdown. El miedo al contagio por contacto físico con billetes empujó a los pasajeros a evitar el efectivo, sumado a que los viajeros esenciales (negocios, salud) tienden a pagar con tarjeta.

- **Recuperación (2022–2025):** la cuota de tarjeta se consolidó en niveles **superiores a los pre-pandemia**, mientras que el efectivo continuó su declive. Esta es una tendencia global acelerada por la pandemia: la digitalización del pago se volvió permanente.

**Factores adicionales:**
- NYC TLC instaló terminales de pago obligatorias en todos los taxis desde 2008, pero la pandemia normalizó su uso exclusivo.
- El crecimiento de servicios como Apple Pay / Google Pay redujo la fricción del pago sin contacto.